In [2]:
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("🚀 [Step 2] 텍스트 길이별 맞춤형 청킹 작업을 시작합니다...\n")

# 1. 깨끗하게 정제된 마스터 파일 불러오기
file_path = '/home/spai0723/Bidcoin/bid_master_cleaned.csv'
df = pd.read_csv(file_path)
print(f"✅ 원본 데이터: 총 {len(df)}건\n")

# 2. 수빈님의 규칙을 적용한 커스텀 청킹 함수
def dynamic_chunking(row):
    text = row['텍스트']
    text_len = row['텍스트길이']
    
    # 예외 처리: 텍스트가 없으면 빈 리스트 반환
    if pd.isna(text) or text_len == 0:
        return []
        
    # [규칙 1] 500자 미만: 청킹 없이 그대로
    if text_len < 500:
        return [text]
        
    # [규칙 2] 500 ~ 3,000자
    elif text_len < 3000:
        chunk_size = 512
        chunk_overlap = 50
        
    # [규칙 3] 3,000 ~ 10,000자
    elif text_len < 10000:
        chunk_size = 512
        chunk_overlap = 100
        
    # [규칙 4] 10,000자 이상
    else:
        chunk_size = 1024
        chunk_overlap = 200

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    
    return splitter.split_text(text)

# 3. 데이터프레임에 청킹 함수 적용
print("✂️ 텍스트를 쪼개고 있습니다. 잠시만 기다려주세요...")
df['청크_리스트'] = df.apply(dynamic_chunking, axis=1)

# 4. 쪼개진 리스트를 각각의 행(Row)으로 분리 (데이터 폭발! 💥)
chunked_df = df.explode('청크_리스트').reset_index(drop=True)

# 5. 컬럼 정리 및 새로운 텍스트 길이 계산
chunked_df = chunked_df.rename(columns={'청크_리스트': '청크_텍스트'})
# 원본 '텍스트'와 '텍스트길이' 컬럼은 이제 필요 없으니 삭제하여 메모리 확보
chunked_df = chunked_df.drop(columns=['텍스트', '텍스트길이'])
chunked_df['청크_길이'] = chunked_df['청크_텍스트'].apply(lambda x: len(x) if isinstance(x, str) else 0)

# 6. 최종 청킹 완료 데이터 저장
save_path = '/home/spai0723/Bidcoin/bid_master_chunked.csv'
chunked_df.to_csv(save_path, index=False, encoding='utf-8-sig')

print("-" * 60)
print(f"✨ 청킹 완료! 원본 {len(df)}건의 문서가 총 {len(chunked_df)}개의 조각으로 나누어졌습니다.")
print(f"💾 청킹된 최종 파일이 '{save_path}'에 저장되었습니다!")

# 결과 샘플 확인
display(chunked_df[['파일명', '청크_텍스트', '청크_길이']].head())

🚀 [Step 2] 텍스트 길이별 맞춤형 청킹 작업을 시작합니다...

✅ 원본 데이터: 총 100건

✂️ 텍스트를 쪼개고 있습니다. 잠시만 기다려주세요...
------------------------------------------------------------
✨ 청킹 완료! 원본 100건의 문서가 총 1239개의 조각으로 나누어졌습니다.
💾 청킹된 최종 파일이 '/home/spai0723/Bidcoin/bid_master_chunked.csv'에 저장되었습니다!


,파일명,청크_텍스트,청크_길이
0,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,2024년 특성화 맞춤형 교육환경 구축 – 트랙운영 학사정보시스템 고도화\r\n ...,481
1,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,Ⅳ. 제안안내 사항 - 5\r\n 1. 입찰 및...,508
2,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,"트랙운영 학사정보시스템 고도화\r\n 사업예산 : 130,000,000원 범위 내...",504
3,한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp,학사운영 시스템을 통해 대학 체제 개편에 대한 대응체계 확립\r\n \r\nⅡ \r...,483
4,한국연구재단_2024년 대학산학협력활동 실태조사 시스템(UICC) 기능개선.hwp,제 안 요 청 서\r\n[ 2024년 대학 산학협력활동 실태조사 시스템(UICC) ...,481


# [프로젝트 보고서] 입찰 공고문 RAG 시스템 구축을 위한 텍스트 청킹(Chunking) 최적화

## 1. 프로젝트 개요
- 목적: 방대한 양의 입찰 공고문 텍스트를 AI(LLM 및 Embedding Model)가 효율적으로 처리할 수 있도록 최적의 크기(Chunk)로 분할

- 핵심 과제: 문서 길이에 따른 가변적 청킹 전략을 수립하고, 문맥 손실을 최소화하는 RecursiveCharacterTextSplitter를 활용하여 검색 효율성 극대화

## 2. 텍스트 길이별 청킹 전략 (EDA 기반)
- 전략 수립: 데이터 탐색 결과, 공고문마다 텍스트 길이 차이가 극심함을 확인하여 문서 길이에 따른 4단계 동적 청킹(Dynamic Chunking) 규칙 적용

| 문서 길이 구간 | 추천 청크 크기 (Size) | 오버랩 (Overlap) | 비고 |
| :--- | :--- | :--- | :--- |
| 500자 미만 | 청킹 없음 (전체 사용) | 0 | 원문 맥락 완전 보존 |
| 500 ~ 3,000자 | 512 tokens | 50 tokens | 단일/이중 청크 생성 |
| 3,000 ~ 10,000자 | 512 tokens | 100 tokens | 중형 문서 문맥 연결 |
| 10,000자 이상 | 1024 tokens | 200 tokens | 대형 문서 정보 밀도 최적화 |


## 3. 구현 로직 및 최적화 기법 (심화)
- 재귀적 분할 (Recursive Splitting): 단순히 글자 수로 자르는 대신, 의미 단위 보존을 위해 \n\n -> \n -> .  ->   순서의 우선순위 구분자(Separators)를 사용하여 자연스러운 문장 끊김 유도

- 데이터 구조 최적화:

    - df.apply()를 통해 행별 맞춤형 스플리터 적용

    - df.explode() 명령어를 사용하여 리스트 형태의 청크들을 각각 독립된 행(Row)으로 펼쳐 검색 가능한 데이터베이스 구조로 변환

- 메모리 관리: 청킹 완료 후 원본 대용량 텍스트 컬럼을 삭제하여 데이터프레임의 경량화 달성

## 4. 데이터 구조 변화 비교 (Transformation)

| 항목 | 전 (Cleaned Data) | 후 (Chunked Data) |
| :--- | :--- | :--- |
| 저장 파일 | `bid_master_cleaned.csv` | `bid_master_chunked.csv` |
| 데이터 단위 | 공고문 파일 1개당 1행 | 공고문 조각(Chunk) 1개당 1행 |
| 주요 컬럼 | `텍스트`, `텍스트길이` | `청크_텍스트`, `청크_길이` |
| 행(Row) 수 | 원본 데이터 수 (N) | 분할된 총 조각 수 (N * M) |


## 5. 결과 분석 및 핵심 인사이트
- 검색 정밀도 확보: 긴 문서를 작은 조각으로 나눔으로써, 추후 RAG 시스템 작동 시 질문과 가장 관련성이 높은 **핵심 정보(Top-K)**만을 선별하여 LLM에 전달 가능

- 문맥 손실 방지: 조각 간의 Overlap(중첩) 구간을 설정하여, 청킹 경계면에서 문장이 잘려 의미가 왜곡되는 현상을 효과적으로 방지함

- 임베딩 효율성: 모든 조각을 모델이 선호하는 512~1024 토큰 내외로 맞춤으로써, 임베딩 벡터 생성 시 정보 압축 손실을 최소화하고 모델의 이해도를 높임